# World Bank `wbgapi` Probe

This notebook is for manually testing the legacy V1-style World Bank flow against the current SQLite holdings data.

If `wbgapi` is not installed in the project venv yet, install it first:

```bash
./venv/bin/pip install wbgapi
```


In [ ]:
from pathlib import Path
import importlib.util
import sqlite3
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "pystocks.sqlite").exists():
    ROOT = ROOT.parent

sys.path.append(str(ROOT))
DB_PATH = ROOT / "data" / "pystocks.sqlite"
DB_PATH


In [ ]:
wbgapi_installed = importlib.util.find_spec("wbgapi") is not None
print("wbgapi installed:", wbgapi_installed)
if not wbgapi_installed:
    raise ImportError("Install wbgapi in the venv before running the rest of this notebook.")

import wbgapi as wb


In [ ]:
from pystocks.preprocess.supplementary import _normalize_economy_code, WORLD_BANK_INDICATOR_MAP

with sqlite3.connect(DB_PATH) as conn:
    holdings_countries = pd.read_sql_query(
        """
        SELECT DISTINCT COALESCE(country_code, country) AS economy_code
        FROM holdings_investor_country
        WHERE COALESCE(country_code, country) IS NOT NULL
        ORDER BY 1
        """,
        conn,
    )

holdings_countries["normalized"] = holdings_countries["economy_code"].map(_normalize_economy_code)
holdings_countries.head(20)


In [ ]:
wb_economies = {row["id"] for row in wb.economy.list()}

normalized = sorted(
    {
        str(code).upper()
        for code in holdings_countries["normalized"].dropna().tolist()
        if len(str(code)) == 3 and str(code).isalpha()
    }
)
recognized = [code for code in normalized if code in wb_economies]
unrecognized = [code for code in normalized if code not in wb_economies]

print("normalized codes:", len(normalized))
print("recognized by World Bank:", len(recognized))
print("unrecognized:", unrecognized)


In [ ]:
years = range(2018, 2027)
indicators = list(WORLD_BANK_INDICATOR_MAP)
chunk_size = 40

frames = []
errors = []
for start in range(0, len(recognized), chunk_size):
    chunk = recognized[start : start + chunk_size]
    try:
        frame = wb.data.DataFrame(indicators, chunk, time=years, labels=False)
        frames.append(frame)
        print(f"chunk {start // chunk_size + 1}: ok ({len(chunk)} economies)")
    except Exception as exc:
        errors.append({"chunk": chunk, "error": repr(exc)})
        print(f"chunk {start // chunk_size + 1}: failed -> {exc}")

len(frames), len(errors)


In [ ]:
errors[:3]


In [ ]:
if not frames:
    raise RuntimeError("No data returned from World Bank via wbgapi.")

raw = pd.concat(frames)
raw


In [ ]:
long = (
    raw.stack(level=list(range(raw.columns.nlevels)))
    .rename("value")
    .reset_index()
)
long.columns = ["economy_code", "series", "year", "value"]
long["feature_name"] = long["series"].map(WORLD_BANK_INDICATOR_MAP)
long.sort_values(["economy_code", "series", "year"]).head(20)


In [ ]:
summary = (
    long.groupby("series", dropna=False)
    .agg(rows=("value", "size"), non_null=("value", lambda s: int(s.notna().sum())))
    .sort_index()
)
summary
